## Terminology Update

Some historical output logs in this notebook were captured before the current naming migration.
Current runtime terminology in the codebase is:
- `single_csv`
- `multi_csv`
- `ExecutionContext`


In [1]:
# Setup: add project root
import sys
import os
sys.path.insert(0, '..')

from src.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS
from src.context.context_factory import create_context
BASE = os.path.abspath(os.path.join('..'))

import logging

# Setup logging for Jupyter notebook
# Force reconfigure by removing existing handlers
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers to avoid duplicates
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

# Add a fresh StreamHandler that outputs to stdout (visible in notebook)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

STANDARD_NAME = "croissant_standard_subset"
# STANDARD_NAME = "spatial_ecological"

print("Imports OK")

Imports OK


In [2]:
from src.orchestrator import run_metadata_extraction
multi_source = {
    'biota': os.path.join(BASE, 'data/biota_dataset/biota.csv'),
    'samples': os.path.join(BASE, 'data/biota_dataset/samples.csv'),
    'species': os.path.join(BASE, 'data/biota_dataset/species.csv'),
}

# multi_source = {
#     'event': os.path.join(BASE, 'data/protoDT/annual_budburst_per_tree.csv'),
#     'occurrence': os.path.join(BASE, 'data/protoDT/budburst_climwin_input.csv'),
#     'temp': os.path.join(BASE, 'data/protoDT/temp_climwin_input.csv'),
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/cropharvest/cropharvest_cleaned_global.csv')
# }

# multi_source = {
#     'event': os.path.join(BASE, 'data/S2BMS/bms_presence_y-2018-2019_th-200.csv'),
# }

In [3]:
data_context = create_context(
    source=multi_source,
    name='biota_multi'
)
# metadata_standard=METADATA_STANDARDS['spatial_ecological']
metadata_standard = METADATA_STANDARDS[STANDARD_NAME]
orchestrator = Orchestrator(topology_name='single')
plan = orchestrator.generate_plan(
    context=data_context,
    metadata_standard=metadata_standard,
    metadata_standard_name=STANDARD_NAME,
)

/tmp/ipykernel_94640/3760421393.py:7: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name='single')


2026-06-24 17:41:58,727 - INFO - root - PlanExecutor initialized with topology: single
2026-06-24 17:41:58,728 - INFO - root -   Players per step: 1
2026-06-24 17:41:58,728 - INFO - root -   Debate rounds: 0
2026-06-24 17:41:58,729 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-06-24 17:41:58,729 - INFO - root - Orchestrator initialized with topology: single
2026-06-24 17:41:58,729 - INFO - root - [ui] planning
2026-06-24 17:41:58,730 - INFO - root - ============================================================
2026-06-24 17:41:58,730 - INFO - root - GENERATING PLAN
2026-06-24 17:41:58,730 - INFO - root - Context: biota_multi
2026-06-24 17:41:58,730 - INFO - root - Context type: multi_csv
2026-06-24 17:41:58,730 - INFO - root - Classified planning type: multi_csv
2026-06-24 17:41:58,730 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-06

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:865: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-06-24 17:42:02,699 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-24 17:42:02,724 - WARNING - root - Plan validation warning: Step 3 ('Generate the final Croissant metadata JSON object using all prior analysis artifacts.') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-06-24 17:42:02,725 - INFO - root - Plan generated successfully!
2026-06-24 17:42:02,725 - INFO - root - Number of steps: 4
2026-06-24 17:42:02,725 - INFO - root -   Step 1: Capture column structures and first-row samples for all resources to establish schema and data types. (player: dataset_schema_preview)
2026-06-24 17:42:02,726 - INFO - root -   Step 2: Identify primary keys, foreign keys, and cross-resource relationships to define data lineage and linkage. (player: relationship_analyst)
2026-06-24 17:42:02,726 - INFO - root -   Step 3: Detect and analyze spatial and temporal columns to determine coverage extent and resolution. (p

In [4]:
# Validate plan dataflow
from src.orchestrator.utils import validate_plan_dataflow

if plan:
    # Convert to dict list for validation
    plan_dicts = plan.to_dict_list()
    
    is_valid, message = validate_plan_dataflow(plan_dicts)
    
    if is_valid:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")
else:
    print("No plan to validate.")

✅ Plan dataflow is valid.


In [5]:
plan_dicts

[{'task': 'Capture column structures and first-row samples for all resources to establish schema and data types.',
  'player': 'dataset_schema_preview',
  'rationale': 'This is a Croissant task. The schema preview provides the necessary column names, data types, and sample values required to define RecordSets and CroissantFields accurately.',
  'target_resources': [],
  'inputs': {},
  'outputs': ['schema_preview']},
 {'task': 'Identify primary keys, foreign keys, and cross-resource relationships to define data lineage and linkage.',
  'player': 'relationship_analyst',
  'rationale': "Relationships (biota->samples, biota->species) are critical for defining the 'references' property in CroissantFields and understanding the dataset structure.",
  'target_resources': [],
  'inputs': {'schema_preview': 'schema_preview'},
  'outputs': ['relationships']},
 {'task': 'Detect and analyze spatial and temporal columns to determine coverage extent and resolution.',
  'player': 'spatial_temporal_sp

In [6]:
plan.pretty_print()

Plan Steps:
Step 0: Capture column structures and first-row samples for all resources to establish schema and data types.
  Rationale: This is a Croissant task. The schema preview provides the necessary column names, data types, and sample values required to define RecordSets and CroissantFields accurately.
  Required Artifacts: {}
  Produced Artifacts: ['schema_preview']
Step 1: Identify primary keys, foreign keys, and cross-resource relationships to define data lineage and linkage.
  Rationale: Relationships (biota->samples, biota->species) are critical for defining the 'references' property in CroissantFields and understanding the dataset structure.
  Required Artifacts: {'schema_preview': 'schema_preview'}
  Produced Artifacts: ['relationships']
Step 2: Detect and analyze spatial and temporal columns to determine coverage extent and resolution.
  Rationale: The 'samples' resource contains 'date' and likely coordinate fields (implied by 'sampling_station_id' and typical marine biolo

In [7]:
from src.orchestrator.plan_executor import PlanExecutor
from src.tools.context_tools import register_context

context_key = "ctx_biota_multi"
register_context(context_key, data_context)
executor = PlanExecutor(topology_name="single")
result = executor.execute(
    plan=plan,
    context=data_context,
    context_key=context_key,
    metadata_standard=metadata_standard,
    metadata_standard_name=STANDARD_NAME
)

2026-06-24 17:42:02,758 - INFO - root - PlanExecutor initialized with topology: single
2026-06-24 17:42:02,759 - INFO - root -   Players per step: 1
2026-06-24 17:42:02,759 - INFO - root -   Debate rounds: 0
2026-06-24 17:42:02,759 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-06-24 17:42:02,759 - INFO - root - Using structured output schema: CroissantStandardSubsetMetadata
2026-06-24 17:42:02,760 - INFO - root - ============================================================
2026-06-24 17:42:02,760 - INFO - root - STARTING PLAN EXECUTION
2026-06-24 17:42:02,760 - INFO - root - Context: biota_multi
2026-06-24 17:42:02,760 - INFO - root - Type: multi_csv
2026-06-24 17:42:02,760 - INFO - root - Resources: ['biota', 'samples', 'species']
2026-06-24 17:42:02,760 - INFO - root - Steps: 4
2026-06-24 17:42:02,760 - INFO - root - ===============================

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:865: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-06-24 17:42:03,241 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-24 17:42:03,252 - INFO - root - Player 'dataset_schema_preview_1': LLM tool loop iteration 1 — model requested 1 tool(s): get_columns_with_first_row
2026-06-24 17:42:09,033 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-24 17:42:09,035 - INFO - root - Player 'dataset_schema_preview_1': LLM tool loop finished after 2 iteration(s), 1 tool invocation(s) — model returned final analysis
2026-06-24 17:42:09,036 - INFO - root - Player 'dataset_schema_preview_1': using LLM tool path — 1 tool result(s), 1 trace entry/entries
2026-06-24 17:42:09,036 - INFO - root -   Player 'dataset_schema_preview_1' completed execution
2026-06-24 17:42:09,036 - INFO - root -     Output: ## Schema Analysis for biota_multi Context  ### Approach Used `get_columns_with_first_row` with no resource filter to capture all three

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:865: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-06-24 17:42:14,535 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-24 17:42:14,536 - INFO - root - Player 'relationship_analyst_1': LLM tool loop finished after 1 iteration(s), 0 tool invocation(s) — model returned final analysis
2026-06-24 17:42:14,536 - INFO - root - Player 'relationship_analyst_1': using LLM tool path — 0 tool result(s), 0 trace entry/entries
2026-06-24 17:42:14,536 - INFO - root -   Player 'relationship_analyst_1' completed execution
2026-06-24 17:42:14,536 - INFO - root -     Output: I'll analyze the relationships between these three resources by examining their schemas, field names, and data patterns. Let me start by getting more detailed information about each resource and then ...
2026-06-24 17:42:14,537 - INFO - root - Single player, skipping debate
2026-06-24 17:42:14,538 - INFO - root - --- STEP 1: SYNTHESIS ---
2026-06-24 17:42:16,994 - INFO - httpx - HTTP Request: POST https://willma.surf.nl

/home/com3dian/Github/metadata_agent/notebooks/../src/players/player.py:865: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-06-24 17:42:44,697 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-06-24 17:42:44,698 - INFO - root - Player 'croissant_metadata_generator_1': LLM tool loop finished after 1 iteration(s), 0 tool invocation(s) — model returned final analysis
2026-06-24 17:42:44,699 - INFO - root - Player 'croissant_metadata_generator_1': using LLM tool path — 0 tool result(s), 0 trace entry/entries
2026-06-24 17:42:44,699 - INFO - root -   Player 'croissant_metadata_generator_1' completed execution
2026-06-24 17:42:44,699 - INFO - root -     Output: {   "name": "Biota monitoring dataset from the Wadden Sea",   "description": "A dataset containing biota abundance and biomass measurements linked to sampling locations, environmental characteristics,...
2026-06-24 17:42:44,699 - INFO - root - Single player, skipping debate
2026-06-24 17:42:44,700 - INFO - root - --- STEP 3: SYNTHESIS ---
2026-06-24 17:42:44,701 - INFO - root -   Using structured

In [8]:
from pprint import pprint
pprint(result.final_workspace['metadata_output'])

{'description': 'A dataset containing biota abundance and biomass measurements '
                'linked to sampling locations, environmental characteristics, '
                'and species taxonomy in the Wadden Sea region.',
 'filesets': {'excludes': [],
              'includes': ['biota.csv', 'samples.csv', 'species.csv']},
 'inLanguage': ['en'],
 'keywords': ['biota',
              'Wadden Sea',
              'benthic',
              'abundance',
              'biomass',
              'AFDM',
              'tidal flat',
              'ecology'],
 'name': 'Biota monitoring dataset from the Wadden Sea',
 'recordsets': [{'examples': ['sample_id: 33941, sampling_station_id: 59, '
                              'sampling_type: grid, date: 2008-07-14, '
                              'platform: boat, tidal_basin_name: Marsdiep, '
                              'tidal_flat_name: Vlakte van Kerken, x: '
                              '4.902667, y: 53.089333, median_grain_size: '
              

In [9]:
executor.list_tools_called()

['get_columns_with_first_row',
 'detect_temporal_columns',
 'detect_spatial_columns',
 'get_temporal_extent',
 'get_spatial_extent',
 'analyze_temporal_column',
 'analyze_spatial_column',
 'analyze_spatial_column']

In [10]:
executor.find_tool_calls('get_temporal_extent')

[{'agent': 'spatial_temporal_specialist_1',
  'input': {'context_key': 'ctx_biota_multi',
   'resource': 'samples',
   'time_column': 'date'},
  'output': {'resource': 'samples',
   'time_column': 'date',
   'total_records': 51851,
   'valid_timestamps': 51851,
   'null_timestamps': 0,
   'temporal_extent': {'start': '2008-07-14 00:00:00',
    'end': '2021-10-26 00:00:00',
    'duration_days': 4852},
   'records_by_year': {2008: 3792,
    2009: 3989,
    2010: 3966,
    2011: 3846,
    2012: 3849,
    2013: 3839,
    2014: 4176,
    2015: 4137,
    2016: 4168,
    2017: 2831,
    2018: 2274,
    2019: 4203,
    2020: 2366,
    2021: 4415},
   'records_by_month': {6: 15431, 7: 18214, 8: 12304, 9: 4449, 10: 1453}}}]

In [11]:
pprint(result.final_workspace['spatial_temporal_analysis'])

('{\n'
 '  "resources": [\n'
 '    {\n'
 '      "name": "samples",\n'
 '      "spatial": {\n'
 '        "coverage": {\n'
 '          "bbox": [\n'
 '            4.801667,\n'
 '            52.895,\n'
 '            7.234333,\n'
 '            53.533333\n'
 '          ]\n'
 '        },\n'
 '        "coordinate_system": "WGS84 (EPSG:4326)",\n'
 '        "resolution": "~1 meter (6 decimal places)",\n'
 '        "columns": {\n'
 '          "longitude": "x",\n'
 '          "latitude": "y"\n'
 '        }\n'
 '      },\n'
 '      "temporal": {\n'
 '        "coverage": {\n'
 '          "from": "2008-07-14",\n'
 '          "to": "2021-10-26"\n'
 '        },\n'
 '        "resolution": "Daily",\n'
 '        "columns": {\n'
 '          "date": "date"\n'
 '        }\n'
 '      }\n'
 '    }\n'
 '  ]\n'
 '}')


In [12]:
print(list(result.final_workspace.keys()))


['metadata_standard', 'schema_preview', 'relationships', 'spatial_temporal_analysis', 'metadata_output']
